# Construction Site Operations — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Construction industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

---

### Pattern for Other Industries
Copy this notebook and adapt the table schemas, column names, thresholds, and
example queries for your industry. Name it `{Industry}_Agent_Instructions.ipynb`.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK (loads INDUSTRY, WAREHOUSE_NAME, etc.)
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Construction Site Operations
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about construction project operations, safety inspections,
documentation burden, subcontractor satisfaction, and production quality using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables (use for lookups and joins):
   - dim_supervisors: supervisor_id, name, role, certifications, years_experience, project_id, hire_date, department, region, email, phone, status
   - dim_projects: project_id, name, type, value, start_date, est_completion, owner, gc_name, status, contract_type, city, state
   - dim_project_sites: site_id, project_id, address, region, active_trades, total_area_sqft, floors, site_status, lat, lon
   - dim_subcontractors: subcontractor_id, name, trade, license_no, safety_rating, projects_completed
   - dim_trade_phases: phase_id, name, project_id, sequence, planned_start, planned_end, budget
   - dim_inspection_types: inspection_type_id, name, category, required_frequency, avg_duration_min, checklist_items

   Batch fact tables (use for metrics and aggregation):
   - fact_daily_logs: log_id, supervisor_id, project_id, date, weather, crew_count, work_summary, doc_time_min, photos_attached, safety_incidents
   - fact_safety_inspections: inspection_id, supervisor_id, site_id, date, type, findings_count, critical_count, doc_time_min, corrective_actions
   - fact_inspection_quality: quality_id, inspection_id, supervisor_id, date, completeness_pct, photo_coverage_pct, defect_identification_rate
   - fact_project_performance: perf_id, project_id, supervisor_id, month, budget_spent, budget_remaining, schedule_variance_days, rfi_count, change_order_count
   - fact_supervisor_wellness: survey_id, supervisor_id, date, overtime_hours, admin_burden_score, fatigue_score, work_life_balance
   - fact_subcontractor_satisfaction: survey_id, subcontractor_id, project_id, date, coordination_score, payment_timeliness, communication_rating

   Event fact tables (also in Eventhouse for real-time):
   - fact_rfi_events: rfi_id, supervisor_id, project_id, timestamp, question, status, response_time_hours, doc_time_min, trade
   - fact_change_orders: change_order_id, project_id, supervisor_id, timestamp, description, cost_impact, schedule_impact_days, doc_time_min
   - fact_phase_handoffs: handoff_id, project_id, from_trade, to_trade, timestamp, punch_list_items, doc_time_min, approved
   - fact_safety_alerts: alert_id, site_id, supervisor_id, timestamp, alert_type, severity, description, immediate_action, osha_reportable
   - fact_pm_interactions: interaction_id, supervisor_id, timestamp, system, module, action, duration_ms, document_type
   - fact_site_location: ping_id, supervisor_id, timestamp, site_id, zone, floor, activity_type, dwell_minutes

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse with pre-built measures and relationships.

3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.

4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (Kusto Query Language).
   Event tables: rfi_events, change_orders, phase_handoffs, safety_alerts, pm_interactions, site_location
   Streaming tables: daily_log_activity, equipment_telemetry, rfi_status, safety_events, weather_conditions

== KEY RELATIONSHIPS ==
- dim_supervisors.supervisor_id joins to fact tables on supervisor_id
- dim_projects.project_id joins to fact tables and dim_project_sites on project_id
- dim_project_sites.site_id joins to fact_safety_inspections, fact_safety_alerts on site_id
- dim_subcontractors.subcontractor_id joins to fact_subcontractor_satisfaction on subcontractor_id
- dim_trade_phases.phase_id references project_id for phase tracking
- dim_inspection_types.inspection_type_id for inspection categorization

== EXAMPLE QUERIES ==

Q: Which supervisors have the highest documentation burden?
→ Query fact_supervisor_wellness ordered by admin_burden_score DESC, join dim_supervisors for names.

Q: What is the average daily log documentation time by project?
→ Query fact_daily_logs, GROUP BY project_id, AVG(doc_time_min), join dim_projects for names.

Q: Which projects are behind schedule?
→ Query fact_project_performance WHERE schedule_variance_days > 0, join dim_projects for details.

Q: Show me safety inspection findings by site.
→ Query fact_safety_inspections, GROUP BY site_id, SUM(findings_count), SUM(critical_count), join dim_project_sites.

Q: How many RFIs are open per project?
→ Query fact_rfi_events WHERE status = 'open', GROUP BY project_id, COUNT(*), join dim_projects.

Q: What is the average change order cost impact by project type?
→ Join fact_change_orders with dim_projects on project_id, GROUP BY type, AVG(cost_impact).

Q: Which subcontractors have the best satisfaction scores?
→ Query fact_subcontractor_satisfaction, GROUP BY subcontractor_id, AVG(coordination_score), join dim_subcontractors.

Q: List supervisors at burnout risk (fatigue > 8).
→ Query fact_supervisor_wellness WHERE fatigue_score > 8, join dim_supervisors.

Q: What is the inspection quality completeness by supervisor?
→ Query fact_inspection_quality, GROUP BY supervisor_id, AVG(completeness_pct), join dim_supervisors.

Q: Which projects have the most change orders?
→ Query fact_change_orders, GROUP BY project_id, COUNT(*), SUM(cost_impact), join dim_projects.
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Construction Site Operations
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on safety compliance, project schedule,
budget tracking, and workforce wellness.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis and trend detection.
   Key operational tables:
   - fact_daily_logs: log_id, supervisor_id, project_id, date, weather, crew_count, work_summary, doc_time_min, photos_attached, safety_incidents
     → Monitor doc_time_min (>60 = warning, >90 = critical), safety_incidents > 0
   - fact_safety_inspections: inspection_id, supervisor_id, site_id, date, type, findings_count, critical_count, doc_time_min, corrective_actions
     → Flag critical_count > 0, track corrective action backlogs
   - fact_inspection_quality: quality_id, inspection_id, supervisor_id, date, completeness_pct, photo_coverage_pct, defect_identification_rate
     → SLA: completeness_pct > 90%, photo_coverage > 80%
   - fact_project_performance: perf_id, project_id, supervisor_id, month, budget_spent, budget_remaining, schedule_variance_days, rfi_count, change_order_count
     → Alert on schedule_variance_days > 5 or budget_remaining < 10% of total
   - fact_supervisor_wellness: survey_id, supervisor_id, date, overtime_hours, admin_burden_score, fatigue_score, work_life_balance
     → CRITICAL: fatigue_score > 8 or admin_burden_score > 8 = burnout risk
   - fact_subcontractor_satisfaction: survey_id, subcontractor_id, project_id, date, coordination_score, payment_timeliness, communication_rating
     → Alert on any score < 5.0
   - fact_rfi_events: rfi_id, supervisor_id, project_id, timestamp, question, status, response_time_hours, doc_time_min, trade
     → Flag response_time_hours > 72 or status = 'overdue'
   - fact_change_orders: change_order_id, project_id, supervisor_id, timestamp, description, cost_impact, schedule_impact_days, doc_time_min
     → Track high cost_impact changes, schedule_impact_days > 7
   - fact_phase_handoffs: handoff_id, project_id, from_trade, to_trade, timestamp, punch_list_items, doc_time_min, approved
     → Flag unapproved handoffs, punch_list_items > 10
   - fact_safety_alerts: alert_id, site_id, supervisor_id, timestamp, alert_type, severity, description, immediate_action, osha_reportable
     → Priority: osha_reportable = true, severity = 'High'
   - fact_pm_interactions: interaction_id, supervisor_id, timestamp, system, module, action, duration_ms, document_type
     → Detect excessive system time (duration_ms > 30000)
   - fact_site_location: ping_id, supervisor_id, timestamp, site_id, zone, floor, activity_type, dwell_minutes
     → Track supervisor coverage across zones

   Dimension tables for context: dim_supervisors, dim_projects, dim_project_sites, dim_subcontractors, dim_trade_phases, dim_inspection_types

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Event tables: rfi_events, change_orders, phase_handoffs, safety_alerts, pm_interactions, site_location
   Streaming tables:
   - daily_log_activity: log_id, supervisor_id, timestamp, project_id, section, entry_type, photo_count
   - equipment_telemetry: telemetry_id, equipment_id, site_id, timestamp, utilization_pct, fuel_level, maintenance_flag
     → Alert on maintenance_flag = true or fuel_level < 15
   - rfi_status: rfi_id, project_id, timestamp, status, assigned_to, days_open, priority
     → Flag days_open > 5 with priority = 'High'
   - safety_events: event_id, site_id, timestamp, event_type, zone, severity, worker_count
     → Immediate alert on severity = 'High'
   - weather_conditions: reading_id, site_id, timestamp, temp_f, wind_mph, precip_in, lightning_flag
     → Alert on lightning_flag = true, wind_mph > 35, temp_f > 105 or < 20

== ALERTING THRESHOLDS ==
- Burnout Risk: fatigue_score > 8, admin_burden_score > 8, overtime_hours > 15
- Safety: critical_count > 0, osha_reportable = true, severity = 'High'
- Schedule: schedule_variance_days > 5 (warning), > 10 (critical)
- Budget: budget_remaining < 10% (warning), < 5% (critical)
- RFIs: response_time_hours > 72, days_open > 5
- Weather: lightning_flag, wind > 35 mph, extreme temps
- Doc Burden: doc_time_min > 60 (warning), > 90 (critical)

== EXAMPLE QUERIES ==

Q: Which supervisors are at burnout risk?
→ Query fact_supervisor_wellness WHERE fatigue_score > 8 OR admin_burden_score > 8, join dim_supervisors.

Q: Are any projects significantly over budget or behind schedule?
→ Query fact_project_performance WHERE schedule_variance_days > 5, join dim_projects.

Q: What open safety alerts need immediate attention?
→ Query fact_safety_alerts WHERE severity = 'High' OR osha_reportable = true.

Q: Which RFIs are overdue?
→ KQL: rfi_status | where days_open > 5 and priority == 'High' | project project_id, status, days_open

Q: Are there dangerous weather conditions at any site?
→ KQL: weather_conditions | where lightning_flag == true or wind_mph > 35

Q: Which equipment needs maintenance?
→ KQL: equipment_telemetry | where maintenance_flag == true | project equipment_id, site_id, utilization_pct

Q: Show projects with the most change orders.
→ Query fact_change_orders, GROUP BY project_id, COUNT(*), SUM(cost_impact), join dim_projects.

Q: Which subcontractors have low satisfaction scores?
→ Query fact_subcontractor_satisfaction WHERE coordination_score < 5 OR communication_rating < 5, join dim_subcontractors.

Q: What is the inspection quality trend?
→ Query fact_inspection_quality, GROUP BY month, AVG(completeness_pct), AVG(photo_coverage_pct).

Q: How much after-hours documentation are supervisors doing?
→ Query fact_supervisor_wellness, ORDER BY overtime_hours DESC, join dim_supervisors.
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all construction site operations data as SQL tables.

DIMENSION TABLES:
- dim_supervisors: supervisor_id, name, role, certifications, years_experience, project_id, hire_date, department, region
- dim_projects: project_id, name, type, value, start_date, est_completion, owner, gc_name, status, contract_type, city, state
- dim_project_sites: site_id, project_id, address, region, active_trades, total_area_sqft, floors, site_status
- dim_subcontractors: subcontractor_id, name, trade, license_no, safety_rating, projects_completed
- dim_trade_phases: phase_id, name, project_id, sequence, planned_start, planned_end, budget
- dim_inspection_types: inspection_type_id, name, category, required_frequency, avg_duration_min, checklist_items

FACT TABLES:
- fact_daily_logs, fact_safety_inspections, fact_inspection_quality, fact_project_performance,
  fact_supervisor_wellness, fact_subcontractor_satisfaction

EVENT FACT TABLES:
- fact_rfi_events, fact_change_orders, fact_phase_handoffs, fact_safety_alerts, fact_pm_interactions, fact_site_location

KEY JOINS:
- dim_supervisors.supervisor_id → fact tables on supervisor_id
- dim_projects.project_id → fact tables and dim_project_sites on project_id
- dim_project_sites.site_id → fact_safety_inspections, fact_safety_alerts on site_id
- dim_subcontractors.subcontractor_id → fact_subcontractor_satisfaction""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- rfi_events, change_orders, phase_handoffs, safety_alerts, pm_interactions, site_location

STREAMING TABLES:
- daily_log_activity: log_id, supervisor_id, timestamp, project_id, section, entry_type, photo_count
- equipment_telemetry: telemetry_id, equipment_id, site_id, timestamp, utilization_pct, fuel_level, maintenance_flag
- rfi_status: rfi_id, project_id, timestamp, status, assigned_to, days_open, priority
- safety_events: event_id, site_id, timestamp, event_type, zone, severity, worker_count
- weather_conditions: reading_id, site_id, timestamp, temp_f, wind_mph, precip_in, lightning_flag

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_supervisor_wellness: CRITICAL if fatigue_score > 8 or admin_burden_score > 8
- fact_safety_inspections: Alert on critical_count > 0
- fact_safety_alerts: Priority on osha_reportable = true, severity = 'High'
- fact_project_performance: schedule_variance_days > 5 (warning), budget_remaining < 10% (warning)
- fact_daily_logs: doc_time_min > 60 (warning), > 90 (critical)
- fact_rfi_events: response_time_hours > 72
- fact_inspection_quality: completeness_pct < 90%

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time telemetry.

STREAMING ALERTS:
- equipment_telemetry: maintenance_flag == true, fuel_level < 15
- safety_events: severity == 'High'
- weather_conditions: lightning_flag == true, wind_mph > 35, extreme temps
- rfi_status: days_open > 5 with priority == 'High'

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which supervisors have the highest documentation burden?",
            "query": """SELECT s.supervisor_id, s.name, s.role,\n       w.admin_burden_score, w.fatigue_score, w.overtime_hours, w.work_life_balance\nFROM fact_supervisor_wellness w\nJOIN dim_supervisors s ON w.supervisor_id = s.supervisor_id\nORDER BY w.admin_burden_score DESC"""
        },
        {
            "question": "What is the average daily log documentation time by project?",
            "query": """SELECT p.name AS project, p.type,\n       COUNT(*) AS log_count,\n       AVG(dl.doc_time_min) AS avg_doc_time,\n       AVG(dl.photos_attached) AS avg_photos\nFROM fact_daily_logs dl\nJOIN dim_projects p ON dl.project_id = p.project_id\nGROUP BY p.name, p.type\nORDER BY avg_doc_time DESC"""
        },
        {
            "question": "Which projects are behind schedule?",
            "query": """SELECT p.name, p.type, p.status,\n       pp.schedule_variance_days, pp.budget_spent, pp.budget_remaining,\n       pp.rfi_count, pp.change_order_count\nFROM fact_project_performance pp\nJOIN dim_projects p ON pp.project_id = p.project_id\nWHERE pp.schedule_variance_days > 0\nORDER BY pp.schedule_variance_days DESC"""
        },
        {
            "question": "Show me safety inspection findings by site.",
            "query": """SELECT ps.address, ps.site_status,\n       COUNT(*) AS inspections,\n       SUM(si.findings_count) AS total_findings,\n       SUM(si.critical_count) AS total_critical\nFROM fact_safety_inspections si\nJOIN dim_project_sites ps ON si.site_id = ps.site_id\nGROUP BY ps.address, ps.site_status\nORDER BY total_critical DESC"""
        },
        {
            "question": "How many RFIs are open per project?",
            "query": """SELECT p.name AS project,\n       COUNT(*) AS open_rfis,\n       AVG(r.response_time_hours) AS avg_response_hours\nFROM fact_rfi_events r\nJOIN dim_projects p ON r.project_id = p.project_id\nWHERE r.status = 'open'\nGROUP BY p.name\nORDER BY open_rfis DESC"""
        },
        {
            "question": "What is the average change order cost impact by project type?",
            "query": """SELECT p.type,\n       COUNT(*) AS change_orders,\n       AVG(co.cost_impact) AS avg_cost_impact,\n       AVG(co.schedule_impact_days) AS avg_schedule_impact\nFROM fact_change_orders co\nJOIN dim_projects p ON co.project_id = p.project_id\nGROUP BY p.type\nORDER BY avg_cost_impact DESC"""
        },
        {
            "question": "Which subcontractors have the best satisfaction scores?",
            "query": """SELECT sc.name, sc.trade,\n       AVG(ss.coordination_score) AS avg_coordination,\n       AVG(ss.payment_timeliness) AS avg_payment,\n       AVG(ss.communication_rating) AS avg_communication\nFROM fact_subcontractor_satisfaction ss\nJOIN dim_subcontractors sc ON ss.subcontractor_id = sc.subcontractor_id\nGROUP BY sc.name, sc.trade\nORDER BY avg_coordination DESC"""
        },
        {
            "question": "What is the inspection quality completeness by supervisor?",
            "query": """SELECT s.name, s.role,\n       AVG(iq.completeness_pct) AS avg_completeness,\n       AVG(iq.photo_coverage_pct) AS avg_photo_coverage,\n       AVG(iq.defect_identification_rate) AS avg_defect_rate\nFROM fact_inspection_quality iq\nJOIN dim_supervisors s ON iq.supervisor_id = s.supervisor_id\nGROUP BY s.name, s.role\nORDER BY avg_completeness ASC"""
        },
        {
            "question": "Which trade phases are running over budget?",
            "query": """SELECT tp.name AS phase, p.name AS project,\n       tp.budget, tp.planned_start, tp.planned_end\nFROM dim_trade_phases tp\nJOIN dim_projects p ON tp.project_id = p.project_id\nORDER BY tp.budget DESC"""
        },
        {
            "question": "Show me phase handoff punch list items by project.",
            "query": """SELECT p.name AS project,\n       ph.from_trade, ph.to_trade,\n       ph.punch_list_items, ph.doc_time_min, ph.approved\nFROM fact_phase_handoffs ph\nJOIN dim_projects p ON ph.project_id = p.project_id\nORDER BY ph.punch_list_items DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which supervisors are at burnout risk?",
            "query": """SELECT s.name, s.role,\n       w.admin_burden_score, w.fatigue_score, w.overtime_hours\nFROM fact_supervisor_wellness w\nJOIN dim_supervisors s ON w.supervisor_id = s.supervisor_id\nWHERE w.fatigue_score > 8 OR w.admin_burden_score > 8\nORDER BY w.fatigue_score DESC"""
        },
        {
            "question": "What are the top 5 projects by value?",
            "query": """SELECT name, type, value, status, city, state\nFROM dim_projects\nORDER BY value DESC\nLIMIT 5"""
        },
        {
            "question": "Show all safety alerts that are OSHA reportable.",
            "query": """SELECT sa.alert_type, sa.severity, sa.description, sa.immediate_action,\n       ps.address AS site\nFROM fact_safety_alerts sa\nJOIN dim_project_sites ps ON sa.site_id = ps.site_id\nWHERE sa.osha_reportable = true"""
        },
        {
            "question": "Which projects have the most RFIs?",
            "query": """SELECT p.name, p.type,\n       COUNT(*) AS rfi_count,\n       AVG(r.response_time_hours) AS avg_response\nFROM fact_rfi_events r\nJOIN dim_projects p ON r.project_id = p.project_id\nGROUP BY p.name, p.type\nORDER BY rfi_count DESC"""
        },
        {
            "question": "What is the average crew count by project?",
            "query": """SELECT p.name,\n       AVG(dl.crew_count) AS avg_crew,\n       COUNT(*) AS log_days\nFROM fact_daily_logs dl\nJOIN dim_projects p ON dl.project_id = p.project_id\nGROUP BY p.name\nORDER BY avg_crew DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Are there any high-priority RFIs overdue?",
            "query": """rfi_status\n| where days_open > 5 and priority == 'High'\n| project project_id, rfi_id, status, days_open, assigned_to"""
        },
        {
            "question": "What equipment needs maintenance?",
            "query": """equipment_telemetry\n| where maintenance_flag == true\n| project equipment_id, site_id, utilization_pct, fuel_level, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "Are there any active safety events?",
            "query": """safety_events\n| where severity == 'High'\n| order by timestamp desc\n| project site_id, event_type, zone, worker_count, timestamp"""
        },
        {
            "question": "What are the current weather conditions at sites?",
            "query": """weather_conditions\n| where lightning_flag == true or wind_mph > 35\n| project site_id, temp_f, wind_mph, precip_in, lightning_flag, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "Show recent daily log activity.",
            "query": """daily_log_activity\n| where timestamp > ago(24h)\n| summarize entries = count(), total_photos = sum(photo_count) by supervisor_id, project_id\n| order by entries desc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which supervisors are at burnout risk right now?",
            "query": """SELECT s.supervisor_id, s.name, s.role,\n       w.admin_burden_score, w.fatigue_score,\n       w.overtime_hours, w.work_life_balance\nFROM fact_supervisor_wellness w\nJOIN dim_supervisors s ON w.supervisor_id = s.supervisor_id\nWHERE w.fatigue_score > 8 OR w.admin_burden_score > 8\nORDER BY w.fatigue_score DESC"""
        },
        {
            "question": "Which projects are significantly behind schedule?",
            "query": """SELECT p.name, p.type, p.status,\n       pp.schedule_variance_days, pp.budget_remaining,\n       pp.rfi_count, pp.change_order_count\nFROM fact_project_performance pp\nJOIN dim_projects p ON pp.project_id = p.project_id\nWHERE pp.schedule_variance_days > 5\nORDER BY pp.schedule_variance_days DESC"""
        },
        {
            "question": "What open safety alerts need immediate attention?",
            "query": """SELECT sa.alert_type, sa.severity, sa.description,\n       sa.immediate_action, sa.osha_reportable,\n       ps.address AS site\nFROM fact_safety_alerts sa\nJOIN dim_project_sites ps ON sa.site_id = ps.site_id\nWHERE sa.severity = 'High' OR sa.osha_reportable = true\nORDER BY sa.timestamp DESC"""
        },
        {
            "question": "Which vendors are underperforming on satisfaction?",
            "query": """SELECT sc.name, sc.trade,\n       AVG(ss.coordination_score) AS avg_coordination,\n       AVG(ss.communication_rating) AS avg_communication\nFROM fact_subcontractor_satisfaction ss\nJOIN dim_subcontractors sc ON ss.subcontractor_id = sc.subcontractor_id\nWHERE ss.coordination_score < 5 OR ss.communication_rating < 5\nGROUP BY sc.name, sc.trade\nORDER BY avg_coordination ASC"""
        },
        {
            "question": "Show projects with the most change orders and cost impact.",
            "query": """SELECT p.name, p.type,\n       COUNT(*) AS change_order_count,\n       SUM(co.cost_impact) AS total_cost_impact,\n       AVG(co.schedule_impact_days) AS avg_schedule_impact\nFROM fact_change_orders co\nJOIN dim_projects p ON co.project_id = p.project_id\nGROUP BY p.name, p.type\nORDER BY total_cost_impact DESC"""
        },
        {
            "question": "Which daily logs have excessive documentation time?",
            "query": """SELECT s.name, p.name AS project,\n       dl.doc_time_min, dl.photos_attached, dl.safety_incidents,\n       CASE WHEN dl.doc_time_min > 90 THEN 'CRITICAL'\n            WHEN dl.doc_time_min > 60 THEN 'WARNING'\n            ELSE 'OK' END AS severity\nFROM fact_daily_logs dl\nJOIN dim_supervisors s ON dl.supervisor_id = s.supervisor_id\nJOIN dim_projects p ON dl.project_id = p.project_id\nWHERE dl.doc_time_min > 60\nORDER BY dl.doc_time_min DESC"""
        },
        {
            "question": "Show overdue RFIs.",
            "query": """SELECT p.name AS project, r.trade,\n       r.response_time_hours, r.status, r.doc_time_min\nFROM fact_rfi_events r\nJOIN dim_projects p ON r.project_id = p.project_id\nWHERE r.status = 'overdue' OR r.response_time_hours > 72\nORDER BY r.response_time_hours DESC"""
        },
        {
            "question": "What inspection quality issues exist?",
            "query": """SELECT s.name, iq.completeness_pct, iq.photo_coverage_pct,\n       iq.defect_identification_rate\nFROM fact_inspection_quality iq\nJOIN dim_supervisors s ON iq.supervisor_id = s.supervisor_id\nWHERE iq.completeness_pct < 90 OR iq.photo_coverage_pct < 80\nORDER BY iq.completeness_pct ASC"""
        },
        {
            "question": "Which phase handoffs have unresolved punch list items?",
            "query": """SELECT p.name AS project,\n       ph.from_trade, ph.to_trade,\n       ph.punch_list_items, ph.approved\nFROM fact_phase_handoffs ph\nJOIN dim_projects p ON ph.project_id = p.project_id\nWHERE ph.approved = false OR ph.punch_list_items > 10\nORDER BY ph.punch_list_items DESC"""
        },
        {
            "question": "How much overtime are supervisors working?",
            "query": """SELECT s.name, s.role,\n       w.overtime_hours, w.admin_burden_score, w.fatigue_score\nFROM fact_supervisor_wellness w\nJOIN dim_supervisors s ON w.supervisor_id = s.supervisor_id\nORDER BY w.overtime_hours DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Are there any active high-severity safety events?",
            "query": """safety_events\n| where severity == 'High'\n| order by timestamp desc\n| project site_id, event_type, zone, worker_count"""
        },
        {
            "question": "What equipment needs urgent maintenance?",
            "query": """equipment_telemetry\n| where maintenance_flag == true or fuel_level < 15\n| project equipment_id, site_id, utilization_pct, fuel_level\n| order by fuel_level asc"""
        },
        {
            "question": "Are there dangerous weather conditions?",
            "query": """weather_conditions\n| where lightning_flag == true or wind_mph > 35 or temp_f > 105 or temp_f < 20\n| project site_id, temp_f, wind_mph, lightning_flag, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "Which RFIs are overdue?",
            "query": """rfi_status\n| where days_open > 5\n| summarize count() by project_id, priority\n| order by count_ desc"""
        },
        {
            "question": "Show recent change orders.",
            "query": """change_orders\n| where timestamp > ago(24h)\n| project project_id, supervisor_id, cost_impact, schedule_impact_days\n| order by cost_impact desc"""
        },
        {
            "question": "What phase handoffs happened today?",
            "query": """phase_handoffs\n| where timestamp > ago(24h)\n| project project_id, from_trade, to_trade, punch_list_items, approved"""
        },
        {
            "question": "Show supervisor site presence today.",
            "query": """site_location\n| where timestamp > ago(24h)\n| summarize total_dwell = sum(dwell_minutes) by supervisor_id, site_id, zone\n| order by total_dwell desc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Construction Data Agents

### QA Agent — `Construction_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Projects | How many active projects do we have? |
| 2 | Projects | What are the top 5 projects by value? |
| 3 | Projects | Which projects are behind schedule? |
| 4 | Projects | Show me project performance by month. |
| 5 | Daily Logs | What is the average daily log documentation time by project? |
| 6 | Daily Logs | Which supervisors log the most safety incidents? |
| 7 | Safety | Show me safety inspection findings by site. |
| 8 | Safety | How many critical safety findings are there per site? |
| 9 | Safety | List all OSHA-reportable safety alerts. |
| 10 | RFIs | How many RFIs are open per project? |
| 11 | RFIs | What is the average RFI response time by trade? |
| 12 | Change Orders | Which projects have the most change orders? |
| 13 | Change Orders | What is the average cost impact by project type? |
| 14 | Supervisors | Which supervisors have the highest admin burden? |
| 15 | Supervisors | List supervisors at burnout risk. |
| 16 | Quality | What is the inspection quality by supervisor? |
| 17 | Quality | Which inspections have low photo coverage? |
| 18 | Subcontractors | Which subcontractors have the best satisfaction scores? |
| 19 | Subcontractors | Show subcontractor payment timeliness ratings. |
| 20 | Phases | Show me phase handoff punch list items by project. |

### Ops Agent — `Construction_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Burnout | Which supervisors are at burnout risk? |
| 2 | Burnout | Who has the most overtime hours? |
| 3 | Safety | What open safety alerts need immediate attention? |
| 4 | Safety | Are there active high-severity safety events? |
| 5 | Safety | Are there dangerous weather conditions at any site? |
| 6 | Schedule | Which projects are significantly behind schedule? |
| 7 | Schedule | Show projects with budget remaining below 10%. |
| 8 | RFIs | Which RFIs are overdue? |
| 9 | Equipment | What equipment needs maintenance? |
| 10 | Equipment | Show equipment with low fuel levels. |
| 11 | Quality | Show inspection quality issues. |
| 12 | Quality | Which inspections have completeness below 90%? |
| 13 | Change Orders | Show projects with the most change orders. |
| 14 | Handoffs | Which phase handoffs have unresolved punch lists? |
| 15 | Subcontractors | Which subcontractors have low satisfaction scores? |
| 16 | Logs | Which daily logs have excessive documentation time? |
| 17 | System | Which supervisors spend the most time in PM systems? |
| 18 | Weather | Are there lightning or high wind alerts? |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")